# Coding Session #3

---

## Today's session

Today's session is structured to introduce pandas. We will explore:

- Reading datasets and basic data exploration
- Recoding variables and if statements

### 0. Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.

In [ ]:
import pandas as pd
import numpy as np

----

### 1. Reading Datasets

This dataset was produced by the Redistricting Data Hub (RDH) using a voter file purchased from L2, a national voter file vendor, on March 9, 2021. The RDH retrieved and processed the data on April 16, 2021.

The original L2 voter file contains individual-level voter records. RDH aggregated these records to the 2010 Census Block level, identified by a GEOID constructed by concatenating County FIPS, Census Tract, and Census Block codes. In this way they were able to ceate counts for categorical variables such as party registration, gender, and modeled commercial data (e.g., likelihood of being a homeowner or magazine subscriptions).

To have a sense of what a Census Block is, you can take a look at this pdf of the [NYC Census Tracts Map](https://www.nyc.gov/assets/planning/download/pdf/about/publications/maps/nyc-census-tracts-map.pdf)



#### Codebook

This table describes the variables included in the dataset. The daset that we use in this coding session is a subset of the original dataset and can be accessesed via [Redistricting Data Hub](https://redistrictingdatahub.org/)

| Variable | Description | Modeled |
|----------|-------------|---------|
| `party_dem` | Count of voters registered with the Democratic Party (L2 Voter File) | No |
| `party_npp` | Count of voters registered with the Non-Partisan Party (L2 Voter File) | No |
| `party_rep` | Count of voters registered with the Republican Party (L2 Voter File) | No |
| `voters_gender_m` | Count of male voters | No |
| `voters_gender_f` | Count of female voters | No |
| `voters_gender_unknkown` | Count of voters with unknown or other gender | No |
| `commercialdatall_gun_owner` | Number of voters in Census Block who own guns based on gun registrations and subscriptions to gun/hunting magazines | No |
| `commercialdatall_home_owner_or_renter_likely_homeowner` | Count of voters likely to be homeowners | Modeled |
| `commercialdatall_home_owner_or_renter_likely_renter` | Count of voters likely to be renters | Modeled |
| `commercialdata_upscalebuyerinhome_avg` | Average number of upscale buyers in the home | Modeled |
| `commercialdata_familymagazineinhome_avg` | Average number of family magazines in the home | Modeled |
| `commercialdata_femaleorientedmagazineinhome_avg` | Average number of female-oriented magazines in the home | Modeled |
| `commercialdata_financialmagazineinhome_avg` | Average number of financial magazines in the home | Modeled |
| `commercialdata_gardeningmagazineinhome_avg` | Average number of gardening magazines in the home | Modeled |
| `commercialdata_healthfitnessmagazineinhome_avg` | Average number of health and fitness magazines in the home | Modeled |







#### 1.1 Load the data

Data usually comes as files (CSV, Excel, SQL, JSON, …). We tell pandas to read a CSV into a DataFrame with `pd.read_csv()`.

In Google Colab, your working directory is in a tempory directory on the remote server. This means that we need to first upload the file we want to use to the server.

1. Look at the **folder icon** on the left sidebar of Colab.
2. Open it, and you’ll see the current content on the server.
3. You can upload files by dragging them into that sidebar or using the `Upload` button.



In [ ]:
# Example: load a CSV called "nyvoterfile_2021.csv" in /content/
df_voterfile = pd.read_csv("nyvoterfile_2021.csv")

# Quick check
df_voterfile.head()

#### 1.2 Orient yourself

When you load a dataset, your first job is orient yourself:

- How many rows/columns are there? (`.shape`)
- What are the column names? (`.columns`)
- What kind of data is in each column (numbers, text, dates)? (`.dtypes`)

These commands are like "peeking inside the box" before doing any analysis.

The attribute `df.shape` tells you the **dimensions of your DataFrame**.  
It returns a tuple `(rows, columns)`:

- The first number = how many rows (observations).  
- The second number = how many columns (variables).  

In [ ]:
df_voterfile.shape

The attribute `df.columns` returns the **names of all the columns** in your DataFrame.  


In [ ]:
df_voterfile.columns

The attribute `df.dtypes` shows the **data type** (`dtype`) of each column in your DataFrame.  

Common pandas dtypes include:  
- **int64** → whole numbers (e.g., counts, IDs)  
- **float64** → decimal numbers (e.g., percentages, continuous values)  
- **object** → text or mixed values (e.g., names, categories)  
- **bool** → True/False values  
- **datetime64** → dates and times  

In [ ]:
df_voterfile.dtypes

When you run `df.info()` in pandas, it gives you a **summary of your DataFrame**.  
This is useful for quickly checking the structure of your dataset.

In [ ]:
df_voterfile.info()

The command `df.describe()` gives you **descriptive statistics** for all the numeric columns in your DataFrame.  
It’s a quick way to understand the basic properties of your data.

In [ ]:
df_voterfile.describe()

###

----

### 2. Recoding variables and if statements

#### 2.1 Math opertations with pandas

In pandas, you can do math directly on columns. Here we are adding together three columns — `party_dem`, `party_rep`, and `party_npp` — to create a new variable called **`total_voters`**.

In [ ]:
total_voters = df_voterfile["party_dem"] + df_voterfile["party_rep"] + df_voterfile["party_npp"]
total_voters.head()

Once we have the **total number of voters** in each row, we can calculate what share (percentage) belongs to each party. We do this by dividing the count of each party by the total:

In [ ]:
pct_dem = df_voterfile["party_dem"] / total_voters
pct_rep = df_voterfile["party_rep"] / total_voters
pct_npp = df_voterfile["party_npp"] / total_voters


We can calculate the **average share of each party** across all geoids. To do this, we use NumPy’s `np.mean()` function, which computes the mean of a Series or array.


In [ ]:
avg_dem = np.mean(pct_dem)
avg_rep = np.mean(pct_rep)
avg_npp = np.mean(pct_npp)

print("Average Democratic Reg. share:", avg_dem)
print("Average Republican Reg. share:", avg_rep)
print("Average Non-partisan Reg. share:", avg_npp)

#### 2.2 Recoding with `np.where` (if statement)

We can use `np.where()` to create a new variable that marks rows where the proportion of non-partisan voters is **above a threshold** (for example, 0.8 = 80%).

In [ ]:
df_voterfile["high_npp"] = np.where(pct_npp > 0.8, 1, 0)

- pct_npp > 0.8: This is the condition. It checks, for each row, whether the proportion of non-partisan voters (pct_npp) is greater than 0.8 (that is, 80%). The result is a column of True or False values.
- np.where(condition, 1, 0): np.where() replaces True with 1 and False with 0.
So every row that meets the condition gets a 1 (high non-partisan area), otherwise a 0.

#### 2.3 Freqency table



Let's now use value_counts() to create a simple frequency table

In [ ]:
df_voterfile["high_npp"].value_counts()

We can also normalise the count and have percentage instead of absolute counts

In [ ]:
# Percentages in %
df_voterfile["high_npp"].value_counts(normalize=True) * 100

Now that we have the column `high_npp` (1 = very high, 0 = not high), we can filter the DataFrame to only keep the rows where `high_npp` equals **1**.



In [ ]:
df_high_npp = df_voterfile[df_voterfile["high_npp"] == 1]
df_high_npp.head(40)

#### 2.4 Subsetting a dataframe (using geoid)

Let’s now zoom into one of these small geographical areas with a high share of non-partisan voters. Doing so will give us a more concrete sense of the data we are working with and how these identifiers connect to real places.

For instance, we can select the row where the GEOID equals 361031697013036. To do this, we filter the DataFrame by that condition:

In [ ]:
# Filter for one specific GEOID using base pandas
geoid_row = df_voterfile[df_voterfile["geoid"] == 361031697013036]
# Show the result
geoid_row

In [ ]:
# Filter for one specific GEOID using querry
geoid_row = df_voterfile.query("geoid == 361031697013036")
# Show the result
geoid_row

In [ ]:
# Filter for one specific GEOID using .loc
geoid_row = df_voterfile.loc[df_voterfile["geoid"] == 361031697013036]
# Show the result
geoid_row

After having done that, let's check out the geoid `361031697013036` using the [Census Interactive Map](https://tigerweb.geo.census.gov/tigerweb/).


1. Open the map and click the search icon in the top right corner.
2. In the dropdown, select Map Layers → Census Tracts and Blocks → 2020 Census Block.
3. Copy and paste the geoid value (361031697013036) into the search box.
4. Hit Enter and the map will zoom directly to that block.

This way you can visually inspect the area that the GEOID represents.

### 3. Exercise: more complex if statements

We want to create a new column called `majority_party` that tells us which group has the **largest share of voters** in each row: Democrats, Republicans, or Non-partisans.

You can use np.where to accomplish this. Since you want to assign values based on multiple conditions, you’ll need to nest several np.where statements. Each np.where works like an if…else check: it evaluates a condition, assigns a value if the condition is true, and otherwise falls back to another option. By nesting them, you can chain together multiple if…elif…else rules. To do this correctly, you’ll also have to combine logical operators (like &) when writing your conditions


In [ ]:

df_voterfile["majority_party"] = np.where(
    (pct_dem > pct_rep) & (pct_dem > pct_npp), "Dem",
    np.where(
        (pct_rep > pct_dem) & (pct_rep > pct_npp), "Rep",
        np.where(
            (pct_npp > pct_dem) & (pct_npp > pct_rep), "NPP",
            "Tie"
        )
    )
)


# Check distribution
df_voterfile["majority_party"].value_counts(normalize=True)

1. **First condition:**  
   `(pct_dem > pct_rep) & (pct_dem > pct_npp)`  
   - This checks if the Democratic share is **bigger than both Republican and Non-partisan shares**.  
   - If true: label the row `"Dem"`.  

2. **Second condition:**  
   `(pct_rep > pct_dem) & (pct_rep > pct_npp)`  
   - Only runs if the first condition was false.  
   - Checks if Republicans have the largest share.  
   - If true: label the row `"Rep"`.  

3. **Third condition:**  
   `(pct_npp > pct_dem) & (pct_npp > pct_rep)`  
   - Runs if both earlier conditions were false.  
   - Checks if Non-partisans have the largest share.  
   - If true: label the row `"NPP"`.  

4. **Default (else):**  
   `"Tie"`  
   - If none of the conditions are true (e.g., two or more groups have the same max value), we assign `"Tie"`.  


# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [ ]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)